In [3]:
from MDAnalysis import *
import MDAnalysis.analysis.distances as distances
import numpy as np
import matplotlib.pyplot as plt
import sys

#set the pdb topology and dcd trajectory
topology = "start_drudes.pdb"
trajectory = "FV_NVT.dcd"

# load trajectory into Universe
u = Universe(topology, trajectory)
total_frame = len(u.trajectory)

# contact distance with electrodes
r_electrode_contact = np.array([ 6.0 , 6.0 , 6.0 , 6.0 ] )

# maximum z distance for pulling BF4 ions, this should be farther than the electrode extends in z...
z_max = 55
rdf_max = 15 # maximum distance to compute rdf
n_bins = 75 # rdf bins
dr = float(rdf_max) / float(n_bins)

# frame to start, may want to skip frames for equilibration
framestart=total_frame // 2
frameend=len(u.trajectory)-1

# set the atoms for computing RDF
resname1="trf"
atoms1='Cf'
resname2="BMI"
atoms2=('C1','C2','C21')

#define the environment classifiers.  These should be strings for atom selections...
classifier=[]
classifier.append( "segid A" )
classifier.append( "segid B" )
classifier.append( "segid C" )
classifier.append( "segid D" )
classifier.append( "segid E" )

# print some info about the trajectory
print("Universe", u, "\n")
print("Atoms", u.atoms, "\n")
print("Residues", u.residues, "\n")
print("Segments", u.segments, "\n")
print("Number of frames in trajectory:", len(u.trajectory), "\n")
print("Time step between frames:", u.trajectory.dt, "fs\n")
print("Total simulation time:", f"{len(u.trajectory) * u.trajectory.dt / 1000:.2f} ns\n")
print("Classifiers:", classifier, "\n")
print("Total frames to analyze:", frameend - framestart + 1, "\n")
print("Frame range:", framestart, "to", frameend, "\n")
print("Contact distances to electrodes:", r_electrode_contact, "\n")

Universe <Universe with 26484 atoms> 

Atoms <AtomGroup [<Atom 1: C0 of type C of resname grp, resid 1 and segid A and altLoc >, <Atom 2: C1 of type C of resname grp, resid 1 and segid A and altLoc >, <Atom 3: C2 of type C of resname grp, resid 1 and segid A and altLoc >, ..., <Atom 26482: DOs of type  of resname trf, resid 400 and segid J and altLoc >, <Atom 26483: DOs1 of type  of resname trf, resid 400 and segid J and altLoc >, <Atom 26484: DOs2 of type  of resname trf, resid 400 and segid J and altLoc >]> 

Residues <ResidueGroup [<Residue grp, 1>, <Residue nan, 1>, <Residue nan, 1>, ..., <Residue trf, 398>, <Residue trf, 399>, <Residue trf, 400>]> 

Segments <SegmentGroup [<Segment A>, <Segment B>, <Segment C>, <Segment D>, <Segment E>, <Segment F>, <Segment G>, <Segment H>, <Segment I>, <Segment J>]> 

Number of frames in trajectory: 2569 

Time step between frames: 10.000000029814057 fs

Total simulation time: 25.69 ns

Classifiers: ['segid A', 'segid B', 'segid C', 'segid D', '

In [4]:
# make static atom groups for electrode.  These don't change from frame-to-frame because electrode is fixed
u.trajectory[0]
electrode_groups=[]
for electrode in classifier:
    electrode_groups.append( u.select_atoms(electrode) )

# frame to start, may want to skip frames for equilibration
framestart=total_frame // 2
frameend=len(u.trajectory)-1

# rdf datastructures, 5 environments
environments_counts=2
rdf_count = [ [ 0 for i in range(n_bins) ] for j in range(environments_counts) ]
rdf_N_N = [ 0 for j in range(2) ] # keeps track of number of particles histogramed for normalization

# trfl/trfl
rdf_count2 = [ [ 0 for i in range(n_bins) ] for j in range(environments_counts) ]
rdf_N_N2 = [ 0 for j in range(2) ] # keeps track of number of particles histogramed for normalization

# loop over trajectory
for t0 in range(framestart, frameend):
    #print( "trajectory frame " , t0 )
    u.trajectory[t0]

# create frame-specific atom groups, note these will change from frame-to-frame
    group1 =  u.select_atoms("name %s and resname %s and prop z < %s" % (atoms1, resname1, z_max) )
    group2 = u.select_atoms("name XXX")
    for ele in atoms2:
        group2 = group2 +  u.select_atoms("name %s and resname %s and prop z < %s" % (ele, resname2, z_max + rdf_max ) )

    # print("group1")
    # print( group1.indices )
    # print("group2")
    # print( group2.indices )

print("electrode_groups", electrode_groups)
print("rdf_count", rdf_count)
print("rdf_count length", len(rdf_count))
print("rdf_N_N", rdf_N_N , "\n")
print("rdf_count2", rdf_count2)
print("rdf_count2 length", len(rdf_count2))
print("rdf_N_N2", rdf_N_N2 , "\n")
print("group1", group1)
print("group1 length", len(group1), "\n")
print("group2", group2)
print("group2 length", len(group2))

electrode_groups [<AtomGroup with 800 atoms>, <AtomGroup with 720 atoms>, <AtomGroup with 720 atoms>, <AtomGroup with 800 atoms>, <AtomGroup with 801 atoms>]
rdf_count [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]
rdf_count length 2
rdf_N_N [0, 0] 

rdf_count2 [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,